## 开源数据集

In [ ]:

import random, json as _json
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
from IPython.display import display, HTML
plt.rcParams['font.family'] = ['Noto Sans CJK SC', 'DejaVu Sans']  # CJK 优先，避免中文豆腐块
plt.rcParams['axes.unicode_minus'] = False
random.seed(42)
np.random.seed(42)

BASE = Path('../data/datasets/danbooru2024')   # 写死 danbooru2024
assert BASE.is_dir(), f'数据集不存在: {BASE.resolve()}'
con = duckdb.connect()
con.execute('SET threads=8')
print(f'数据集: {BASE.resolve()}')

## 字段字典与枚举值含义

danbooru2024 只有一个元数据文件 `metadata.parquet`：约 806 万行 Danbooru post 元数据（一行一图）。含义列对 88 个字段逐一注释；枚举含义按实测分布写死。


In [ ]:

META_FILE = 'metadata.parquet'   # danbooru2024 唯一元数据文件
p = BASE / META_FILE
assert p.is_file(), f'文件不存在: {p.resolve()}'

DESC = {   # 字段 → 含义（数据字典）
    'id': 'post 主键 ID', 'created_at': '上传时间', 'updated_at': '最后更新时间',
    'uploader_id': '上传者 ID', 'approver_id': '审核人 ID（未审核=null）',
    'score': '总分 = up_score - down_score', 'up_score': '顶数', 'down_score': '踩数', 'fav_count': '收藏数',
    'source': '原始来源 URL（pixiv 等）', 'pixiv_id': '关联 pixiv ID（无=null）',
    'md5': '图片内容 md5', 'file_ext': '文件扩展名', 'file_size': '文件大小（字节）',
    'image_width': '图片宽(px)', 'image_height': '图片高(px)',
    'rating': '分级：g/s/q/e（含义见下方枚举表）',
    'tag_string': '全部标签（空格分隔，带类型前缀）', 'tag_count': '标签总数',
    'tag_count_general': '一般标签数', 'tag_count_artist': '画师标签数',
    'tag_count_character': '角色标签数', 'tag_count_copyright': '作品标签数', 'tag_count_meta': '元标签数',
    'tag_string_general': '一般标签串', 'tag_string_character': '角色标签串',
    'tag_string_copyright': '作品标签串', 'tag_string_artist': '画师标签串', 'tag_string_meta': '元标签串',
    'parent_id': '父 post（系列/变体图，无=null）', 'has_children': '是否有子 post',
    'has_active_children': '是否有未删除的子 post', 'has_visible_children': '是否有可见子 post',
    'has_large': '是否有大图版本',
    'is_pending': '待审核', 'is_flagged': '被标记删除申请', 'is_deleted': '已删除', 'is_banned': '被封禁',
    'last_commented_at': '最后评论时间', 'last_comment_bumped_at': '最后评论 bump 时间',
    'last_noted_at': '最后翻译注释时间', 'bit_flags': '内部位标记',
    'file_url': '原图直链', 'large_file_url': '大图直链', 'preview_file_url': '预览图直链',
}

def desc_of(name):
    if name in DESC: return DESC[name]
    if name.startswith('media_asset.variants.'):
        i = name.split('.')[2]
        return f'第{i}组变体: ' + {'type': '类型(preview/180x180/original)', 'url': '直链', 'width': '宽(px)',
                                  'height': '高(px)', 'file_ext': '扩展名'}[name.split('.')[3]]
    return {'media_asset.id': '资产 ID', 'media_asset.created_at': '资产创建时间',
            'media_asset.updated_at': '资产更新时间', 'media_asset.md5': '资产内容 md5',
            'media_asset.file_ext': '资产扩展名', 'media_asset.file_size': '资产大小(字节)',
            'media_asset.image_width': '资产宽(px)', 'media_asset.image_height': '资产高(px)',
            'media_asset.duration': '视频时长(秒，静态图=null)', 'media_asset.status': '资产状态(实测全 active)',
            'media_asset.file_key': 'URL 文件键', 'media_asset.is_public': '是否公开(实测全 True)',
            'media_asset.pixel_hash': '像素级哈希'}.get(name, '')

schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p}')").fetchall()
n = con.execute(f"SELECT count(*) FROM read_parquet('{p}')").fetchone()[0]
print(f'{p.name}: {n:,} 行 x {len(schema)} 字段')
with pd.option_context('display.max_rows', 100, 'display.max_colwidth', 80, 'display.width', 200):
    display(pd.DataFrame([{'field': c, 'type': t, '含义': desc_of(c)} for c, t, *_ in schema]))

# 枚举值含义（按实测分布写死）
print('rating      g=general 全年龄(27.6%)  s=sensitive 轻度敏感(52.6%)  q=questionable 擦边(11.3%)  e=explicit 露骨(8.6%)')
print('file_ext    jpg 73.5% png 25.3% | 动态/打包: mp4 0.4% gif 0.4% zip 0.2%(ugoira) webp 0.1% webm swf avif')
print('media_asset.status  全量 active（快照即活跃 post，无 expunged/deleted 态）')
print('variants.type       180x180=预览缩略图（其余槽位为 sample 等缩放档；swf 无缩略图故为 original）')


## 单字段下钻

换文件/换字段：改下方 cell 开头的 `META_FILE` 与 `FIELD`。

In [ ]:

META_FILE = 'metadata.parquet'   # ← 改这里：换元数据文件
FIELD     = 'rating'          # ← 改这里：要下钻的字段名（None=报错列出可选字段）

con.execute('DROP TABLE IF EXISTS vc')
con.execute(f"CREATE OR REPLACE VIEW m AS SELECT * FROM read_parquet('{BASE / META_FILE}')")
cols = [r[0] for r in con.execute("SELECT column_name FROM (DESCRIBE SELECT * FROM m)").fetchall()]
if FIELD is None:
    raise AssertionError(f'请指定 FIELD，可选字段（{len(cols)}）: ' + ', '.join(cols[:40])
                         + (' …' if len(cols) > 40 else ''))
assert FIELD in cols, f'字段 {FIELD!r} 不存在，可选: {", ".join(cols)}'
print(f'元数据: {META_FILE}   字段: {FIELD}')

In [ ]:
   # 'id': 'post 主键 ID', 'created_at': '上传时间', 'updated_at': '最后更新时间',
   # 'uploader_id': '上传者 ID', 'approver_id': '审核人 ID（未审核=null）',
   # 'score': '总分 = up_score - down_score', 'up_score': '顶数', 'down_score': '踩数', 'fav_count': '收藏数',
   # 'source': '原始来源 URL（pixiv 等）', 'pixiv_id': '关联 pixiv ID（无=null）',
   # 'md5': '图片内容 md5', 'file_ext': '文件扩展名', 'file_size': '文件大小（字节）',
   # 'image_width': '图片宽(px)', 'image_height': '图片高(px)',
   # 'rating': '分级：g/s/q/e（含义见下方枚举表）',
   # 'tag_string': '全部标签（空格分隔，带类型前缀）', 'tag_count': '标签总数',
   # 'tag_count_general': '一般标签数', 'tag_count_artist': '画师标签数',
   # 'tag_count_character': '角色标签数', 'tag_count_copyright': '作品标签数', 'tag_count_meta': '元标签数',
   # 'tag_string_general': '一般标签串', 'tag_string_character': '角色标签串',
   # 'tag_string_copyright': '作品标签串', 'tag_string_artist': '画师标签串', 'tag_string_meta': '元标签串',
   # 'parent_id': '父 post（系列/变体图，无=null）', 'has_children': '是否有子 post',
   # 'has_active_children': '是否有未删除的子 post', 'has_visible_children': '是否有可见子 post',
   # 'has_large': '是否有大图版本',
   # 'is_pending': '待审核', 'is_flagged': '被标记删除申请', 'is_deleted': '已删除', 'is_banned': '被封禁',
   # 'last_commented_at': '最后评论时间', 'last_comment_bumped_at': '最后评论 bump 时间',
   # 'last_noted_at': '最后翻译注释时间', 'bit_flags': '内部位标记',
   # 'file_url': '原图直链', 'large_file_url': '大图直链', 'preview_file_url': '预览图直链',
FILTER_SQL = "rating = 'g' and (tag_string_character IS NOT NULL and TRY_CAST(tag_string_character AS VARCHAR) != '') "   # ← 改这里：SQL WHERE 条件（duckdb 语法）

META_FILE = 'metadata.parquet'   # ← 改这里：换元数据文件
FIELD1     = 'tag_string_copyright'  
FIELD2     = 'tag_string_character'          # ← 改这里：要下钻的字段名（None=报错列出可选字段）

con.execute('DROP TABLE IF EXISTS vc')
con.execute(f"CREATE OR REPLACE VIEW m AS SELECT * FROM read_parquet('{BASE / META_FILE}') where {FILTER_SQL}")
cols = [r[0] for r in con.execute("SELECT column_name FROM (DESCRIBE SELECT * FROM m)").fetchall()]

# ---- 取值分布：总数 / 空值 / 取值数 / TopN 覆盖率 / Gini ----
N_ALL = con.execute('SELECT count(*) FROM m').fetchone()[0]
N_NA = con.execute(f"SELECT count(*) FROM m WHERE {FIELD1} IS NULL OR {FIELD2} IS NULL").fetchone()[0]

con.execute(f'''CREATE TABLE vc AS
    SELECT {FIELD1} AS copyright, {FIELD2} AS character, count(*) AS n
    FROM m GROUP BY 1,2''')

vc = con.execute('SELECT copyright, character, n FROM vc ORDER BY n DESC, copyright').df()
N_VC = len(vc)

TOT = vc.n.sum()
cov = lambda k: vc.n.head(k).sum() / TOT * 100

def gini(x):
    x = np.sort(np.asarray(x, float)); n = len(x)
    return (2 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum())) - (n + 1) / n

G = gini(vc.n.values)
print(f'总记录: {N_ALL:,}   空值: {N_NA:,} ({N_NA/max(TOT,1):.1%})   非空: {TOT-N_NA:,}')
print(f'取值总数: {N_VC:,}   Top10: {cov(10):.1f}%   Top100: {cov(100):.1f}%   Top1000: {cov(1000):.1f}%   Gini: {G:.3f}')
q = np.percentile(vc.n.values, [25, 50, 90])
print(f'每取值记录数: p25={q[0]:.0f}  p50={q[1]:.0f}  p90={q[2]:.0f}  最大={vc.n.max():,} ({vc.copyright.iloc[0]})')

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].loglog(np.arange(1, N_VC + 1), vc.n.values, lw=1)
axes[0].set_title(f'{FIELD1} × {FIELD2} 组合长尾（x=按量排名, y=记录数, log-log）')
top30 = vc.head(30)
axes[1].barh(top30.copyright.str.slice(0, 30)[::-1], top30.n.values[::-1])
axes[1].set_title(f'Top30 组合（按 {FIELD1}）')
axes[2].hist(np.log10(np.clip(vc.n.values, 1, None)), bins=50)
axes[2].set_title('每取值记录数分布（log10）')
axes[2].set_xlabel('log10(记录数)')
plt.tight_layout()
plt.show()
display(vc.head(30))


# ---- 均衡化适用性提示（启发式，供人工判断） ----
if N_VC <= 200:
    s = f'取值少（{N_VC:,}），可直接作为均衡化分层字段'
elif N_VC <= 10_000:
    s = f'取值中等（{N_VC:,}），可作均衡化字段；建议低频值（记录数<p25）合并或欠采样'
else:
    s = f'取值过多（{N_VC:,}），不建议直接按原始值均衡，先聚合/分桶（Top-K + 其余归一桶）'
print(f'· {s}')
print(f'· Gini={G:.3f}: ' + ('头部极集中，长尾严重——均衡需欠采样头部或加权采样' if G > 0.9 else
                          '分布偏斜——均衡时注意头部占比' if G > 0.7 else '分布较均衡'))
print(f'· Top1000 覆盖 {cov(1000):.1f}%: ' + ('头部即主体，截断 Top-K 损失小' if cov(1000) > 80 else '长尾占比大，截断会丢较多信息'))

### 抽样看 case

`SAMPLE_VALUES=None` 时自动取记录数 Top6 取值，每个取值抽 `SAMPLES_PER_VALUE` 条；卡片自动渲染 http 图（danbooru 走 preview_file_url CDN 缩略图），其余字段文本展示。

In [ ]:

SAMPLE_VALUES     = None    # ← 改这里：指定要看的取值列表，如 ['hatsune_miku']；None=自动取记录数 Top6
SAMPLES_PER_VALUE = 4       # ← 改这里：每个取值抽几条
SAMPLE_FIELDS     = None    # ← 改这里：卡片显示哪些字段；None=自动（除置顶字段外前 8 个）
SAMPLE_FIELD      = FIELD2  # ← 改这里：按哪个字段置顶抽样（FIELD1 / FIELD2）
SAMPLE_FILTER     = "1=1"   # ← 改这里：抽样时的附加过滤条件（duckdb 语法；叠加在 cell 6 的 FILTER_SQL 之上）；"1=1"=不过滤

# ---- 按 SAMPLE_FIELD 现算取值分布（cell 6 的 vc 是 FIELD1×FIELD2 组合，此处按单字段重聚）----
vc_s = con.execute(
    f'''SELECT TRY_CAST({SAMPLE_FIELD} AS VARCHAR) AS v, count(*) AS n
        FROM m WHERE {SAMPLE_FILTER} GROUP BY 1 ORDER BY n DESC''').df()
vals = SAMPLE_VALUES if SAMPLE_VALUES is not None else vc_s.v[vc_s.v != '(null)'].head(6).tolist()
fields = SAMPLE_FIELDS or [c for c in cols if c != SAMPLE_FIELD][:8]
if SAMPLE_FIELD not in fields:
    fields = [SAMPLE_FIELD] + fields       # 置顶抽样字段

def card(r):
    img_html = ''
    for c in cols:
        v = r.get(c)
        if isinstance(v, str) and (v.startswith('http') or (BASE / v).is_file() or Path('..').joinpath(v).is_file()):
            src = v if v.startswith('http') else (v if (BASE / v).is_file() else str(Path('..') / v))
            img_html = f'<img src="{src}" loading="lazy" style="height:200px;max-width:260px;object-fit:contain;background:#f6f6f6">'
            break
    kv = ''.join(f'<div style="font-size:12px"><b>{c}</b>: {str(r.get(c))[:120]}</div>' for c in fields)
    return f'<td style="vertical-align:top;padding:6px;border:1px solid #ddd">{img_html}{kv}</td>'

for v in vals:
    sub = con.execute(
        f'''SELECT * FROM m
            WHERE {SAMPLE_FILTER}
              AND CASE WHEN {SAMPLE_FIELD} IS NULL OR TRY_CAST({SAMPLE_FIELD} AS VARCHAR) = ''
                       THEN '(null)' ELSE TRY_CAST({SAMPLE_FIELD} AS VARCHAR) END = ?
            LIMIT 4000''', [v]).df().sample(min(SAMPLES_PER_VALUE, 4000), random_state=42)
    if len(sub) == 0:
        print(f'取值不存在: {v}')
        continue
    display(HTML(f'<h4>{SAMPLE_FIELD} = {str(v)[:80]}（全量 {int(vc_s.n[vc_s.v==v].iloc[0]) if (vc_s.v==v).any() else "?"} 条，抽样 {len(sub)}）</h4>'))
    trs = ['<tr>' + ''.join(card(r) for r in sub[i:i+4].to_dict('records')) + '</tr>' for i in range(0, len(sub), 4)]
    display(HTML('<table style="border-collapse:collapse">' + ''.join(trs) + '</table>'))


## 目标数据分布

In [ ]:
M_BASE   = Path('../data/datasets/demiwtg')                    # 写死 demiwtg
M_META   = M_BASE / 'meta' / 'metadata.jsonl'                  # 写死清单路径
SOURCE   = None   # ← 改这里：migrate 部分；None=不限来源
N_SAMPLE = 8                   # ← 改这里：抽几条
FIELDS   = None                # ← 改这里：卡片字段；None=默认（迁移关注面）

mcon = duckdb.connect()


# ---- 读全表到 pandas（后续分布/过滤都基于它；复用部件三的 mcon / M_BASE）----
# ---- 过滤看 case ----
# metadata.jsonl 可用字段：
#   sha256(str)  ext(str)  source(str)  license(str)  author(str|None)
#   content_url(str)  landing_url(str|None)  path(str)  mime(str|None)
#   fetched_at(float)  size_bytes(int)  width(int)  height(int)
#   orig_width(int|None)  orig_height(int|None)
#   instances(list[str])  queries(dict)  query_langs(dict)
#   kb_match(int 0-10)  richness(int)  identity(bool)  focus(int 0-10)
#   quality(float 0-10)  caption(str|None)
FILTER_SQL = "quality >=8.5 and identity=true "   # ← 改这里：SQL WHERE 条件（duckdb 语法）
# 示例：
#   "kb_match >= 5 AND identity = true"
#   "source IN ('pixiv', 'anilist') AND quality >= 7"
#   "width >= 512 AND height >= 512 AND richness >= 5"
#   "instances[1] = '初音未来' AND kb_match >= 6"
#   "1=1"  -- 不过滤，全量
N_CASES = 8                     # ← 改这里：抽几条

mdf = mcon.execute(f"SELECT * FROM read_json_auto('{M_META}', maximum_object_size=1073741824)  WHERE {FILTER_SQL}").df()
print(f'metadata.jsonl 全表 {len(mdf):,} 条')

midf = mcon.execute(f"SELECT instances[1],count(*) FROM read_json_auto('{M_META}', maximum_object_size=1073741824)  WHERE {FILTER_SQL} group by 1").df()
print(f'metadata.jsonl 实例数 {len(midf):,} 条')


SCORE_FIELDS = ['kb_match', 'identity', 'focus', 'quality']   # ← 改这里：要看的打分字段
N_BINS = 21                                                    # ← 改这里：直方图分箱数

# ---- 分布 ----
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, field in zip(axes.ravel(), SCORE_FIELDS):
    vals = mdf[field].dropna()
    n_total = len(mdf)
    n_na = n_total - len(vals)
    if field == 'identity':
        tc = int(vals.sum())
        fc = len(vals) - tc
        ax.bar(['False', 'True'], [fc, tc], color=['#e88', '#8c8'])
        ax.set_title(f'{field}（bool）  True={tc:,}  False={fc:,}  null={n_na:,}')
        for i, v in enumerate([fc, tc]):
            ax.text(i, v + max(fc, tc) * 0.02, f'{v:,}', ha='center', fontsize=10)
    else:
        lo, hi = float(vals.min()), float(vals.max())
        if hi - lo < 1e-9:
            ax.text(0.5, 0.5, f'{field}\n唯一值={lo}', ha='center', va='center', transform=ax.transAxes)
        else:
            bins = np.linspace(lo, hi, N_BINS)
            ax.hist(vals, bins=bins, edgecolor='white', alpha=0.85)
            mean_v, med_v = vals.mean(), vals.median()
            ax.axvline(mean_v, color='red', ls='--', lw=1, label=f'mean={mean_v:.2f}')
            ax.axvline(med_v, color='orange', ls='--', lw=1, label=f'median={med_v:.2f}')
            ax.legend(fontsize=8)
            ax.set_title(f'{field}（{vals.dtype}）  min={lo}  max={hi}  null={n_na:,}')
    ax.set_xlabel('')
plt.tight_layout()
plt.show()


In [ ]:

# ---- 每个数据源的各项得分 + identity 分布 ----
SCORE_COLS = ['kb_match', 'focus', 'quality', 'richness']   # ← 改这里：要对比的打分字段

g = mdf.groupby('source')
stat = pd.DataFrame({
    '条数': g.size(),
    **{c: g[c].mean().round(2) for c in SCORE_COLS},
    'identity%': (100 * g['identity'].apply(lambda s: (s == True).sum())
                  / g['identity'].apply(lambda s: s.notna().sum()).clip(lower=1)).round(1),
})
stat['占比%'] = (100 * stat['条数'] / stat['条数'].sum()).round(1)
stat = stat.sort_values('条数', ascending=False)
with pd.option_context('display.width', 200):
    display(stat)

order = list(stat.index)   # 按条数降序
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, col in zip(axes.ravel()[:4], SCORE_COLS):
    data = [mdf.loc[mdf['source'] == s, col].dropna().values for s in order]
    ax.boxplot(data, vert=False, showfliers=False, widths=0.6)
    ax.set_yticks(range(1, len(order) + 1), order, fontsize=7)
    ax.set_title(f'{col} 按源分布（箱线）')
ax = axes[1][1]   # identity True 率
vals = stat.loc[order[::-1], 'identity%']
ax.barh(range(len(order)), vals, color='#6a9')
ax.set_yticks(range(len(order)), order[::-1], fontsize=7)
for i, v in enumerate(vals):
    ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=7)
ax.set_title('identity=True 占比（按源）')
ax = axes[1][2]   # 条数占比
vals = stat.loc[order[::-1], '条数']
ax.barh(range(len(order)), vals, color='#9ab')
ax.set_yticks(range(len(order)), order[::-1], fontsize=7)
for i, v in enumerate(vals):
    ax.text(v, i, f'{v:,}', va='center', fontsize=7)
ax.set_title('条数（按源）')
plt.tight_layout()
plt.show()


## 多实例与重复检查

In [ ]:
# ---- 多实例与 (图, instance) 重复检查 ----
# metadata.jsonl 为记录驱动迁移产物：每实例×图一行，去重键 (sha256, instance)
N_CASES_MULTI = 1   # ← 改这里：多实例图抽几张
N_CASES_DUP   = 1    # ← 改这里：重复 (图, instance) 键抽几个

FILTER_SQL = "quality >=8.5 and identity=true "  

md = mcon.execute(
    f"SELECT sha256, instances FROM read_json_auto('{M_META}', maximum_object_size=1073741824) where {FILTER_SQL}").df()
pairs = md.explode('instances').rename(columns={'instances': 'instance'})
pairs = pairs[pairs.instance.notna()]
n_rows, n_imgs = len(md), md.sha256.nunique()

per_img = pairs.groupby('sha256')['instance'].agg(['nunique', 'size'])
multi = per_img[per_img['nunique'] > 1]                    # 挂了 2+ 个不同实例的图
key_cnt = pairs.value_counts(['sha256', 'instance'])
dup = key_cnt[key_cnt > 1]                                 # 出现 2+ 次的 (图, instance) 键
n_extra = int(dup.sum() - len(dup))                        # 按去重键算多出来的冗余行

print(f'总行数: {n_rows:,}   去重图片数: {n_imgs:,}   去重 (图,instance) 键数: {len(key_cnt):,}')
print(f'多实例图片: {len(multi):,} ({len(multi)/max(n_imgs,1):.1%} of 图片)')
print('实例数分布: ' + '  '.join(f'{k} 个实例: {v:,} 张' for k, v in multi['nunique'].value_counts().sort_index().items()))
print(f'重复 (图,instance): {len(dup):,} 个键, 冗余行 {n_extra:,} ({n_extra/max(n_rows,1):.1%} of 总行数)')

# ---- 抽样卡片：一图一卡，逐行列出该行 instance 与打分 ----
show_fields = ['instances', 'source', 'kb_match', 'identity', 'focus', 'quality', 'caption']

def cards_of(shas):
    in_list = ','.join(f"'{s}'" for s in shas)
    fsub = mcon.execute(
        f"SELECT * FROM read_json_auto('{M_META}', maximum_object_size=1073741824) WHERE sha256 IN ({in_list}) and {FILTER_SQL}").df()
    for sha, grp in fsub.sort_values('source').groupby('sha256', sort=False):
        r0 = grp.iloc[0]
        ip = M_BASE / r0['path'] if isinstance(r0.get('path'), str) and (M_BASE / r0['path']).is_file() else None
        img = (f'<img src="{ip}" loading="lazy" style="max-height:280px;max-width:380px;object-fit:contain;background:#f6f6f6">'
               if ip else '<div style="color:#c00">blob 缺失</div>')
        rows = ''.join('<div style="border-top:1px dashed #ccc;padding:4px 0">'
                       + ''.join(f'<div style="font-size:12px;margin:1px 0"><b>{c}</b>: {str(r.get(c))[:220]}</div>'
                                 for c in show_fields) + '</div>'
                       for _, r in grp.iterrows())
        display(HTML(f'<div style="display:flex;gap:12px;border:1px solid #ddd;padding:8px;margin:8px 0;align-items:flex-start">'
                     f'{img}<div style="min-width:0"><b>{sha[:16]}…</b>（{len(grp)} 行）{rows}</div></div>'))

if len(multi):
    shas = list(pd.Series(multi.index).sample(min(N_CASES_MULTI, len(multi)), random_state=42))
    print(f'\n===== 多实例图片抽样（{len(shas)} 张）=====')
    cards_of(shas)
else:
    print('\n无多实例图片')

if len(dup):
    keys = dup.index.to_series().sample(min(N_CASES_DUP, len(dup)), random_state=42)
    print(f'\n===== 重复 (图,instance) 抽样（{len(keys)} 键，展示对应图的全部行）=====')
    for k in keys:
        print(f'  {k[0][:16]}… × {k[1]}  → {int(dup.loc[k])} 行')
    cards_of(list(dict.fromkeys(k[0] for k in keys)))
else:
    print('\n无重复 (图,instance)')


In [ ]:
# ---- 每实例图数分布（多图分布；复用 M_META / M_BASE / FILTER_SQL）----
# 口径与 collect_v2 覆盖算子一致：每清单行给行内每个实例各计一张（跨实例不共享）
import json as _json
import duckdb

FILT = (FILTER_SQL or "1=1").strip() or "1=1"
icon = duckdb.connect()          # 独立连接，不污染部件三的 mcon
pairs = icon.execute(f"""
    SELECT unnest(instances) AS instance
    FROM read_json_auto('{M_META}', maximum_object_size=1073741824)
    WHERE {FILT}
""").df()
pairs = pairs[pairs.instance.notna()]
cnt = pairs.groupby('instance').size()

with open(M_BASE / 'meta' / 'instances.json', encoding='utf-8') as f:
    all_names = {i['name'] for i in _json.load(f)['instances']}
n_zero = len(all_names - set(cnt.index))          # 实例表有、清单无 → 0 图

total = len(cnt) + n_zero
med, mean = cnt.median(), cnt.mean()
print(f'实例总数: {total:,}（清单内 {len(cnt):,} + 0 图 {n_zero:,}）')
print(f'每实例图数: 中位 {med:.0f}  均值 {mean:.1f}  max {int(cnt.max())}')
print(f'0 张: {n_zero:,}   仅 1 张: {int((cnt == 1).sum()):,}   <2 张: {n_zero + int((cnt == 1).sum()):,}'
      f'   ≥8 张(--skip-covered 阈值): {int((cnt >= 8).sum()):,}')

bins = pd.cut(cnt, bins=[-1, 0, 1, 7, 10**9],
              labels=['0', '1', '2-7', '>=8']).value_counts().reindex(['0', '1', '2-7', '>=8']).fillna(0)
bins['0'] += n_zero
print('覆盖档分布: ' + '   '.join(f'{k} 张: {v:,}' for k, v in bins.items()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
CAP = 50                       # ← 改这里：线性直方图截断上限（尾部并进末柱）
ax = axes[0]
vals = cnt[cnt <= CAP].values
hi = min(CAP, max(int(vals.max()), 1))
ax.hist(vals, bins=np.arange(0, hi + 2) - 0.5, edgecolor='white', color='#7a9', alpha=0.85)
ax.axvline(8, color='red', ls='--', lw=1, label='skip-covered=8')
ax.legend(fontsize=9)
ax.set_title(f'每实例图数分布（截断 {CAP}，max={int(cnt.max())}）')
ax.set_xlabel('图数')
ax = axes[1]                   # 对数轴看全尾部：长尾/重尾一眼辨
ax.hist(cnt.values, bins=np.logspace(0, np.log10(cnt.max() + 1), 40),
        edgecolor='white', color='#97a', alpha=0.85)
ax.set_xscale('log')
ax.set_title('每实例图数分布（log 轴）')
ax.set_xlabel('图数')
plt.tight_layout()
plt.show()


## 目标数据抽样

In [ ]:
# ---- 过滤看 case ----
# metadata.jsonl 可用字段：
#   sha256(str)  ext(str)  source(str)  license(str)  author(str|None)
#   content_url(str)  landing_url(str|None)  path(str)  mime(str|None)
#   fetched_at(float)  size_bytes(int)  width(int)  height(int)
#   orig_width(int|None)  orig_height(int|None)
#   instances(list[str])  queries(dict)  query_langs(dict)
#   kb_match(int 0-10)  richness(int)  identity(bool)  focus(int 0-10)
#   quality(float 0-10)  caption(str|None)
FILTER_SQL = "quality >= 8 AND identity = true"  
# 示例：
#   "quality >= 8 AND identity = true"
#   "source IN ('pixiv', 'anilist') AND quality >= 7"
#   "width >= 512 AND height >= 512 AND richness >= 5"
#   "instances[1] = '初音未来' AND kb_match >= 6"
#   "1=1"  -- 不过滤，全量
N_CASES = 8                     # ← 改这里：抽几条

fsub = mcon.execute(f"SELECT * FROM read_json_auto('{M_META}', maximum_object_size=1073741824) WHERE {FILTER_SQL}").df()
print(f'过滤: WHERE {FILTER_SQL}  →  命中 {len(fsub):,} 条（展示 {min(N_CASES, len(fsub))} 条）')
fsub = fsub.sample(min(N_CASES, len(fsub)), random_state=42)

show_fields = ['instances', 'source', 'kb_match', 'identity', 'focus', 'quality', 'caption']
for _, r in fsub.iterrows():
    ip = M_BASE / r['path'] if isinstance(r.get('path'), str) and (M_BASE / r['path']).is_file() else None
    img = (f'<img src="{ip}" loading="lazy" style="max-height:280px;max-width:380px;object-fit:contain;background:#f6f6f6">'
           if ip else '<div style="color:#c00">blob 缺失</div>')
    kv = ''.join(f'<div style="font-size:12px;margin:1px 0"><b>{c}</b>: {str(r.get(c))[:220]}</div>' for c in show_fields)
    display(HTML(f'<div style="display:flex;gap:12px;border:1px solid #ddd;padding:8px;margin:8px 0;align-items:flex-start">'
                 f'{img}<div style="min-width:0">{kv}</div></div>'))

## 按 taxonomy 视角

把目标清单的 instance 打标挂回标签体系：挂载关系从 `meta/taxonomy.json` 现场聚合（复用 `taxonomy/mount_map.py`，不落盘）。按树深度聚合看各节点数据量，可限定子树、按节点抽样看图；三种全量导出均默认落 `state/curation/`（置 None 关闭）：`EXPORT_FILE`（按深度聚合表）、`EXPORT_TREE_FILE`（实例清单，节点×实例×图片数）、`EXPORT_TREE_FULL_FILE`（完整树，一行一节点，叶子附合并去重的采集 source 清单）。

In [ ]:
# ---- taxonomy 视角：instance → 挂载路径从 taxonomy.json 现场聚合（mount_map 契约）----
TAX_DEPTH  = 4        # ← 改这里：按哪一层树深度聚合（根=0；越小越粗）
NODE_PATH  = None     # ← 改这里：只看某节点子树，如 '融合世界标签体系 / IP 分类标签 / 虚构角色 IP'；None=整棵树
FILTER_SQL = "quality >= 8 AND identity = true"  
EXPORT_FILE = Path('../state/curation/taxonomy_per_node.csv')   # ← 改这里：全量表导出路径；None=不导出
EXPORT_TREE_FILE = Path('../state/curation/taxonomy_tree_instances.csv')   # ← 改这里：实例清单树（一行一节点，叶子带实例名单）导出；None=不导出
EXPORT_TREE_FULL_FILE = Path('../state/curation/taxonomy_tree_full.csv')   # ← 改这里：完整树（一行一节点，叶子带实例 source 清单）导出；None=不导出

import sys
sys.path.insert(0, str(Path('..').resolve()))
from taxonomy.mount_map import load_mount_map, tree_sibling_of

mounts = load_mount_map(tree_sibling_of(str(M_META)))   # {实例名: [挂载节点路径, ...]}
print(f'taxonomy: {len(mounts):,} 个实例有挂载点')

mdt = mcon.execute(
    f"SELECT sha256, instances FROM read_json_auto('{M_META}', maximum_object_size=1073741824) WHERE {FILTER_SQL}").df()
tp = mdt.explode('instances').rename(columns={'instances': 'instance'})
tp['paths'] = tp['instance'].map(mounts)
unmounted = tp[tp.paths.isna()]
tp = tp[tp.paths.notna()].explode('paths')        # 一处挂载记一行；一图多挂载会重复计入各节点
if NODE_PATH:
    tp = tp[tp.paths.str.startswith(NODE_PATH)]
tp['node'] = tp.paths.str.split(' / ').str[:TAX_DEPTH + 1].str.join(' / ')

per_node = tp.groupby('node').agg(
    行数=('sha256', 'size'), 图片数=('sha256', 'nunique'), 实例数=('instance', 'nunique'))
per_node['占比%'] = (100 * per_node.行数 / max(len(tp), 1)).round(1)
per_node = per_node.sort_values('行数', ascending=False)

from IPython.display import FileLink   # notebook 里生成可点击下载链接（相对 notebook 所在目录）
def _dl_link(p):
    display(FileLink(str(p), result_html_prefix='下载: '))

# ---- 导出全量表格（不止 head(30)；utf-8-sig 供 Excel 直接打开）----
if EXPORT_FILE is not None:
    EXPORT_FILE.parent.mkdir(parents=True, exist_ok=True)
    out = per_node.reset_index().rename(columns={'node': 'node_path'})
    out.insert(1, 'depth', out.node_path.str.count(' / '))
    out.to_csv(EXPORT_FILE, index=False, encoding='utf-8-sig')
    print(f'全量表已导出: {len(out):,} 行 → {EXPORT_FILE.resolve()}')

# ---- 导出实例清单树（与 taxonomy_tree_full.csv 同构：一行一节点；叶子带挂载实例名单，| 分隔）----
if EXPORT_TREE_FILE is not None:
    import json as _json
    tree = _json.load(open(tree_sibling_of(str(M_META)), encoding='utf-8'))['tree']
    rows = []
    def walk(n):
        is_leaf = not (n.get('children') or [])
        insts = [(i.get('name') if isinstance(i, dict) else i) for i in (n.get('instances') or [])]
        insts = [str(i).strip() for i in insts if i is not None and str(i).strip()]
        # 与 taxonomy_tree_full.csv 同构：一行一节点；叶子带挂载实例名单（| 分隔，树序即展示序不去重），非叶子为空
        rows.append((n.get('path', ''),'|'.join(insts) if is_leaf else ''))
        for ch in n.get('children') or []:
            walk(ch)
    walk(tree)
    EXPORT_TREE_FILE.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows, columns=['node_path', 'instance清单']).to_csv(
        EXPORT_TREE_FILE, index=False, encoding='utf-8-sig')
   # print(f'实例清单树已导出: {len(rows):,} 行（一行一节点，树遍历序）→ {EXPORT_TREE_FILE.resolve()}')
   # _dl_link(EXPORT_TREE_FILE)

# ---- 导出完整树（一行 = 一个节点；叶子节点附其挂载实例在 instances.json 的 source 清单）----
if EXPORT_TREE_FULL_FILE is not None:
    import json as _json
    _src = mcon.execute(
        f"SELECT DISTINCT unnest(instances) AS instance, source "
        f"FROM read_json_auto('{M_META}', maximum_object_size=1073741824) WHERE {FILTER_SQL}").df()
    inst_src = _src.groupby('instance')['source'].agg(lambda s: '|'.join(sorted(s.dropna().unique()))).to_dict()
    tree = _json.load(open(tree_sibling_of(str(M_META)), encoding='utf-8'))['tree']
    rows2 = []
    def walk2(n):
        is_leaf = not (n.get('children') or [])
        insts = [(i.get('name') if isinstance(i, dict) else i) for i in (n.get('instances') or [])]
        insts = [str(i).strip() for i in insts if i is not None and str(i).strip()]
        _s = set()
        for nm in insts:                     # 该叶子全部挂载实例的采集 source 合并去重（不带实例名）
            if nm in inst_src:
                _s.update(inst_src[nm].split('|'))
        srcs = '|'.join(sorted(_s))
        rows2.append((n.get('path', ''), srcs if is_leaf else ''))   # 只留 3 列
        for ch in n.get('children') or []:
            walk2(ch)
    walk2(tree)
    EXPORT_TREE_FULL_FILE.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows2, columns=['node_path','source清单']).to_csv(
        EXPORT_TREE_FULL_FILE, index=False, encoding='utf-8-sig')
    #print(f'完整树已导出: {len(rows2):,} 行（一行一节点，叶子带 source 清单）→ {EXPORT_TREE_FULL_FILE.resolve()}')

    n_unm, n_unm_img = len(unmounted), unmounted.sha256.nunique()
    print(f'过滤后行数: {len(mdt):,}   挂载命中: {len(tp):,} 行   未命中: {n_unm:,} 行（{n_unm_img:,} 张，待认领池/死名）')
    print('注：一图多挂载时各节点重复计数，占比合计可能 > 100%')
with pd.option_context('display.max_rows', 50, 'display.width', 200):
    display(per_node.head(100000))
if n_unm:
    display(unmounted.instance.value_counts().head(10).rename_axis('未挂载实例 Top10').to_frame())

In [ ]:
# ---- 节点量级图 + 按节点抽样看图 ----
PLOT_DEPTH = 1          # ← 改这里：条形图聚合深度（根=0；改 1/2/3/4 看各层分布）；默认跟随 TAX_DEPTH
N_TOP_PLOT = 20         # ← 改这里：条形图展示前多少个节点
SAMPLE_NODES = None     # ← 改这里：要抽样看图的节点路径列表（须与 PLOT_DEPTH 对齐）；None=自动取行数 Top3
N_CASES      = 1        # ← 改这里：每个节点抽几张
EXPORT_DEPTH    = 4     # ← 改这里：落盘抽样聚合深度（独立于 PLOT_DEPTH；1/2/3/4 看各层）
EXPORT_PER_NODE = 1     # ← 改这里：落盘抽样每节点抽几张（不足该数则有几张抽几张）
SAMPLE_EXPORT = Path('../state/curation/taxonomy_sample_cases')  # ← 改这里：落盘抽样导出目录（原分辨率图 + metadata.jsonl）；None=不导出
EXPORT_FIELDS = ['sha256', 'instances', 'queries', 'caption']  # ← 改这里：落盘元数据保留字段（精简省 token；可加 'sample_node' 看抽样所属节点）

# 按 PLOT_DEPTH 现场重算节点（只影响本 cell 的图与抽样，不动上一格 per_node 的 TAX_DEPTH 聚合）
_plot_node = tp.paths.str.split(' / ').str[:PLOT_DEPTH + 1].str.join(' / ')
per_plot = tp.groupby(_plot_node).agg(
    行数=('sha256', 'size'), 图片数=('sha256', 'nunique'), 实例数=('instance', 'nunique'))
per_plot['占比%'] = (100 * per_plot.行数 / max(len(tp), 1)).round(1)
per_plot = per_plot.sort_values('行数', ascending=False)

top = per_plot.head(N_TOP_PLOT)
plt.figure(figsize=(10, max(4, len(top) * 0.3)))
plt.barh(range(len(top)), top.行数.values[::-1], color='#8ab')
plt.yticks(range(len(top)), [n.split(' / ')[-1] for n in top.index[::-1]], fontsize=8)
plt.xlabel('行数（实例×图，多挂载重复计）')
plt.title(f'depth≤{PLOT_DEPTH} 节点行数 Top{len(top)}' + (f'（子树: {NODE_PATH}）' if NODE_PATH else ''))
plt.tight_layout(); plt.show()

nodes = SAMPLE_NODES if SAMPLE_NODES is not None else list(per_plot.head(3).index)
show_fields = ['instances', 'source', 'kb_match', 'identity', 'focus', 'quality', 'caption']
for nd in nodes:
    shas_pool = tp.loc[_plot_node == nd, 'sha256'].unique()
    if len(shas_pool) == 0:
        print(f'节点无数据: {nd}'); continue
    rng = np.random.default_rng(42)
    shas = list(rng.choice(shas_pool, size=min(N_CASES, len(shas_pool)), replace=False))
    in_list = ','.join(f"'{s}'" for s in shas)
    sub = mcon.execute(
        f"SELECT * FROM read_json_auto('{M_META}', maximum_object_size=1073741824)"
        f" WHERE sha256 IN ({in_list}) AND {FILTER_SQL}").df()
    print(f'===== 节点: {nd}（池内 {len(shas_pool):,} 张，抽样 {len(sub)} 行）=====')
    for _, r in sub.iterrows():
        ip = M_BASE / r['path'] if isinstance(r.get('path'), str) and (M_BASE / r['path']).is_file() else None
        img = (f'<img src="{ip}" loading="lazy" style="max-height:280px;max-width:380px;object-fit:contain;background:#f6f6f6">'
               if ip else '<div style="color:#c00">blob 缺失</div>')
        kv = ''.join(f'<div style="font-size:12px;margin:1px 0"><b>{c}</b>: {str(r.get(c))[:220]}</div>' for c in show_fields)
        display(HTML(f'<div style="display:flex;gap:12px;border:1px solid #ddd;padding:8px;margin:8px 0;align-items:flex-start">'
                     f'{img}<div style="min-width:0">{kv}</div></div>'))

# ---- 抽样落盘（独立抽样，不依赖上方展示：原分辨率图按 sha256 原样拷贝 + metadata.jsonl）----
if SAMPLE_EXPORT is not None:
    import shutil
    _exp_node = tp.paths.str.split(' / ').str[:EXPORT_DEPTH + 1].str.join(' / ')
    rng = np.random.default_rng(42)
    picks = []                       # (节点, sha256) 抽样对；池内不足 EXPORT_PER_NODE 张则有几张抽几张
    for nd, pool in tp.groupby(_exp_node)['sha256'].unique().items():
        if len(pool):
            picks += [(nd, s) for s in rng.choice(pool, size=min(EXPORT_PER_NODE, len(pool)), replace=False)]
    SAMPLE_EXPORT.mkdir(parents=True, exist_ok=True)
    recs, copied = [], {}
    all_shas = sorted({s for _, s in picks})
    frames = []                      # 分批 IN 查询，避免 SQL 过长、也避免逐节点重扫全表
    for i in range(0, len(all_shas), 500):
        in_list = ','.join(f"'{s}'" for s in all_shas[i:i+500])
        frames.append(mcon.execute(
            f"SELECT * FROM read_json_auto('{M_META}', maximum_object_size=1073741824)"
            f" WHERE sha256 IN ({in_list}) AND {FILTER_SQL}").df())
    by_sha = {s: g for s, g in pd.concat(frames).groupby('sha256')} if frames else {}
    for nd, sha in picks:
        for _, r in by_sha.get(sha, pd.DataFrame()).iterrows():
            d = {k: r[k] for k in EXPORT_FIELDS if k in r and r[k] is not None}
            ip = M_BASE / r['path'] if isinstance(r.get('path'), str) and (M_BASE / r['path']).is_file() else None
            if ip is not None:
                if sha not in copied:             # 同一图多行元数据只拷一份字节
                    dst = SAMPLE_EXPORT / f'{sha}{ip.suffix}'
                    shutil.copy2(ip, dst)
                    copied[sha] = dst.name
                d['sample_image'] = copied[sha]
            recs.append(d)
    with open(SAMPLE_EXPORT / 'metadata.jsonl', 'w', encoding='utf-8') as f:
        for d in recs:
            f.write(_json.dumps(d, ensure_ascii=False, default=str, separators=(',', ':')) + '\n')
    print(f'抽样落盘: depth={EXPORT_DEPTH} 节点 {len(set(n for n, _ in picks)):,} 个 / {len(recs)} 行元数据 / {len(copied)} 张图（原分辨率）→ {SAMPLE_EXPORT.resolve()}')


## 评测集构建（v1）

把抽样落盘目录整理成自包含评测集：图片重命名为「序号_实例名_sha 前 8 位」放入 `images/`，元数据带相对地址 `image` 字段；按 sha256 去重（同一图被多节点抽中只留一条）。整体可打包带走。


In [ ]:

EVAL_SRC    = Path('../state/curation/taxonomy_sample_cases')  # ← 改这里：抽样落盘目录（上一格产物）
EVAL_OUT    = Path('../state/curation/eval_v2')                # ← 改这里：评测集输出目录（images/ + metadata.jsonl）
EVAL_FIELDS = ['sha256', 'instances', 'queries', 'caption']    # ← 改这里：评测集元数据字段（image 字段自动前置；源里有 sample_node 会自动带上）

import re, shutil, ast as _ast
def _clean(s, n=40):
    s = re.sub(r'[\\/:*?"\'<>|\s]+', '_', str(s)).strip('_')
    return s[:n] or 'noname'

(EVAL_OUT / 'images').mkdir(parents=True, exist_ok=True)
rows = [_json.loads(l) for l in open(EVAL_SRC / 'metadata.jsonl', encoding='utf-8') if l.strip()]
seen, recs = {}, []
for r in rows:
    sha = r.get('sha256')
    if not sha or sha in seen:                       # 同一图被多节点抽中/多实例多行：只留一条
        continue
    src_img = next(iter(EVAL_SRC.glob(f'{sha}.*')), None)
    if src_img is None:
        print(f'缺图跳过: {sha[:16]}…'); continue
    insts = r.get('instances')
    if isinstance(insts, str):                       # 旧版导出可能把列表经 default=str 序列化成字符串，先还原
        try: insts = _ast.literal_eval(insts)
        except Exception: pass
    inst = insts[0] if isinstance(insts, list) and insts else insts
    name = f'{len(seen)+1:04d}_{_clean(inst)}_{sha[:8]}{src_img.suffix}'
    shutil.copy2(src_img, EVAL_OUT / 'images' / name)
    d = {'image': f'images/{name}'}                  # 相对评测集根目录的地址
    for k in EVAL_FIELDS:
        v = r.get(k)
        if v is None:
            continue
        if isinstance(v, str) and v[:1] in '[{':     # 还原被序列化成字符串的列表/字典
            try: v = _ast.literal_eval(v)
            except Exception: pass
        d[k] = v
    if r.get('sample_node') is not None:
        d['sample_node'] = r['sample_node']
    seen[sha] = name
    recs.append(d)

with open(EVAL_OUT / 'metadata.jsonl', 'w', encoding='utf-8') as f:
    for d in recs:
        f.write(_json.dumps(d, ensure_ascii=False, default=str, separators=(',', ':')) + '\n')
print(f'评测集 v1: {len(recs)} 条（源 {len(rows)} 行去重后）→ {EVAL_OUT.resolve()}')
display(pd.DataFrame(recs).head(10))
